# Analityka BGK PLN — dług agencyjny Skarbu Państwa

Dashboard dla obligacji BGK wszystkich programów PLN (FPC + KFD + FP + FWSZ + własne), z perspektywą porównawczą do POLGB. Wzór: CETO_DOWNLOADER (`bondspot_dashboard.ipynb`).

**Źródła:**
- `bgk_auctions` — XLSX *Baza obligacji* (kanon wszystkich emisji PLN, public + private)
- `bgk_auction_results` — PDF *Komunikaty* (aukcje FPC publiczne: stop_price, B/C, NK)
- `v_bgk_auction_metrics` — wyliczone B/C, NK shares, allocation rate
- `v_bgk_auction_spread` — spread vs POLGB na aukcji (FPC PDF auctions)
- `v_bgk_issuance_spread` — spread vs POLGB per issuance event (cross-program)
- `v_bgk_outstanding_timeline` — event-driven cumulative outstanding per program × currency
- POLGB benchmark: `polgb_curve_at()` + `polgb_yield_interp()` + `polgb_floater_dm_interp()`

**Konwencja spread:** `spread_bp = (BGK - POLGB) × 100`. Dla yield-space (stałokuponowe) i DM-space (floatery WZ). Dodatnia = BGK płaci więcej = słaba aukcja / koncesja. Ujemna = BGK płaci mniej = silny popyt / przejście "through".


In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

load_dotenv(Path("..") / ".env")

SUPABASE_URL = os.environ["SUPABASE_URL"].rstrip("/")
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

HEADERS = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json",
}

PAGE_SIZE = 1000

START_DATE = pd.Timestamp("2020-04-01")  # FPC start
START_DATE_STR = START_DATE.strftime("%Y-%m-%d")
TODAY = pd.Timestamp.today().normalize()
TODAY_STR = TODAY.strftime("%Y-%m-%d")


def _paginate(method, url, *, json_body=None, timeout=120, max_pages=200):
    rows: list = []
    for page in range(max_pages):
        offset = page * PAGE_SIZE
        h = {**HEADERS,
             "Range-Unit": "items",
             "Range": f"{offset}-{offset + PAGE_SIZE - 1}"}
        kw = {"headers": h, "timeout": timeout}
        if json_body is not None:
            kw["json"] = json_body
        r = requests.request(method, url, **kw)
        if r.status_code not in (200, 206):
            r.raise_for_status()
        chunk = r.json()
        if not isinstance(chunk, list):
            return chunk
        rows.extend(chunk)
        if len(chunk) < PAGE_SIZE:
            return rows
    raise RuntimeError(f"_paginate hit max_pages={max_pages} without finishing")


def rpc(name: str, payload: dict | None = None) -> pd.DataFrame:
    rows = _paginate("POST", f"{SUPABASE_URL}/rest/v1/rpc/{name}",
                     json_body=payload or {})
    return pd.DataFrame(rows)


def fetch_view(name: str, query: str = "?select=*") -> pd.DataFrame:
    rows = _paginate("GET", f"{SUPABASE_URL}/rest/v1/{name}{query}")
    return pd.DataFrame(rows)


# ============================ LLM commentary ============================
# Shared llm_commentary table with CETO (per handoff decision 2). section
# field = "bgk_*" distinguishes from CETO's polgb commentaries.

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
LLM_MODEL = "claude-opus-4-7"
HISTORY_N = 3

_llm_client = None
def _get_llm_client():
    global _llm_client
    if _llm_client is not None:
        return _llm_client
    if not ANTHROPIC_API_KEY:
        return None
    try:
        import anthropic
        _llm_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        return _llm_client
    except ImportError:
        return None


_LLM_SYSTEM_PROMPT = (
    "Jesteś doświadczonym analitykiem rynku polskiego długu publicznego, "
    "specjalizującym się w obligacjach BGK gwarantowanych przez Skarb Państwa "
    "(programy FPC / KFD / FP / FWSZ / własne) oraz benchmarku POLGB. "
    "Piszesz precyzyjnie, profesjonalnie, bez bullshitu, w języku polskim. "
    "Używasz terminologii branżowej (bid-to-cover, spread vs POLGB, discount margin, "
    "WIBOR/POLSTR, tail/through, NK, program-level allocation, FPC ramp-up).\n\n"
    "TWARDE OGRANICZENIA - NIE WOLNO CI:\n"
    "1. Spekulować o oczekiwaniach stóp procentowych (cięcia/podwyżki NBP, EBC, Fed) "
    "ani twierdzić jak rynek się pozycjonuje co do polityki monetarnej - chyba że "
    "konkretne dane (forwards, OIS, ankiety) są w prompcie.\n"
    "2. Cytować/zakładać poziomów inflacji, CPI, PKB, deficytu budżetowego, "
    "potrzeb pożyczkowych itp. których nie ma w danych wejściowych.\n"
    "3. Odwoływać się do sytuacji geopolitycznej, wyborów, ratingu, news flow, "
    "wypowiedzi członków RPP - nie masz tych informacji.\n"
    "4. Używać sformułowań typu 'w środowisku oczekiwań na X', 'rynek wycenia Y', "
    "'sentyment jest Z' bez konkretnego źródła w danych.\n\n"
    "CO WOLNO: interpretować to co widać w danych (popyt vs podaż, spread vs POLGB, "
    "preferencje per coupon kind / per program, struktura WAM portfela, trendy "
    "serii czasowej), tłumaczyć mechanikę (np. 'wysoki NK = real money chciał więcej "
    "niż agent emisji oferował', 'spread BGK over POLGB to premia płynnościowa "
    "agencyjnego ryzyka pomimo gwarancji SP'), oraz odwoływać się do swoich "
    "wcześniejszych analiz tej sekcji jeśli zostały podane. Jeśli dane "
    "nie wystarczają do wniosku - napisz to wprost zamiast zmyślać kontekst."
)


def _fetch_commentary_history(section: str, limit: int = HISTORY_N) -> list:
    try:
        url = f"{SUPABASE_URL}/rest/v1/rpc/llm_commentary_history"
        r = requests.post(url, headers=HEADERS,
                          json={"p_section": section, "p_limit": limit},
                          timeout=15)
        if r.status_code != 200:
            return []
        data = r.json()
        return data if isinstance(data, list) else []
    except Exception:
        return []


def _save_commentary(section, chart_name, prompt, response,
                      input_tokens, output_tokens):
    try:
        url = f"{SUPABASE_URL}/rest/v1/llm_commentary"
        payload = {
            "snapshot_date": TODAY_STR,
            "section": section,
            "chart_name": chart_name,
            "model": LLM_MODEL,
            "prompt": prompt,
            "response": response,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
        }
        r = requests.post(url, headers={**HEADERS, "Prefer": "return=minimal"},
                          json=payload, timeout=20)
        if r.status_code >= 400:
            print(f"  ! save_commentary HTTP {r.status_code}: {r.text[:200]}", flush=True)
    except Exception as exc:
        print(f"  ! save_commentary error: {exc}", flush=True)


def _build_history_block(section: str) -> str:
    hist = _fetch_commentary_history(section)
    if not hist:
        return ""
    lines = ["", "=== POPRZEDNIE TWOJE ANALIZY TEJ SEKCJI (od najnowszej) ==="]
    for h in hist:
        lines.append(f"\n[{h.get('snapshot_date', '?')}]\n{h.get('response', '')}")
    lines.append("=== END HISTORII ===\n")
    return "\n".join(lines)


def llm_chart_commentary(chart_name: str, what_it_shows: str, data_summary: str,
                          max_tokens: int = 350) -> str:
    client = _get_llm_client()
    if client is None:
        return ("> ⚠️ _Brak ANTHROPIC_API_KEY lub `anthropic` package - "
                "LLM commentary wyłączony._")
    section = "bgk_" + chart_name.split("—", 1)[0].strip().lower().replace(" ", "_")
    history_block = _build_history_block(section)
    prompt = (
        f"WYKRES: **{chart_name}**\n\n"
        f"CO POKAZUJE: {what_it_shows}\n\n"
        f"DANE TERAZ:\n{data_summary}\n"
        + history_block +
        "\nNapisz **krótki** (2-4 zdania) komentarz analityczny w języku polskim. "
        "Bez bullet points, bez powtarzania surowych liczb - skup się na "
        "interpretacji, kontekście historycznym i sygnale rynkowym."
    )
    try:
        response = client.messages.create(
            model=LLM_MODEL,
            max_tokens=max_tokens,
            system=_LLM_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": prompt}],
        )
        text = response.content[0].text.strip()
        _save_commentary(
            section=section, chart_name=chart_name, prompt=prompt, response=text,
            input_tokens=getattr(response.usage, "input_tokens", None),
            output_tokens=getattr(response.usage, "output_tokens", None),
        )
        return f"> 💬 **Analiza:** {text}"
    except Exception as exc:
        return f"> ⚠️ _LLM error: {exc}_"


def llm_final_report(context: str, max_tokens: int = 900) -> str:
    client = _get_llm_client()
    if client is None:
        return ("⚠️ _Brak ANTHROPIC_API_KEY lub `anthropic` package - "
                "finalny raport wyłączony._")
    section = "bgk_final_auction_report"
    history_block = _build_history_block(section)
    prompt = (
        "Otrzymujesz pelny kontekst dashboardu + dzisiejszej aukcji BGK. "
        "Napisz profesjonalny raport koncowy (250-350 slow) w jezyku polskim.\n\n"
        "**Wymagana struktura:**\n"
        "1. **TL;DR** (1-2 linie: weak/typowo/strong + glowne uzasadnienie)\n"
        "2. **Demand** - co napedzilo/utrudnilo popyt, vs historia per seria\n"
        "3. **Spread vs POLGB** - czy BGK platy premie czy poszedl through, kontekst\n"
        "4. **Program mix** - jak dzisiejszy print wpisuje sie w cross-program "
        "structure (FPC dominacja vs KFD/FP/FWSZ)\n"
        "5. **Funding context** - YTD pace per program, wnioski dla nastepnych tygodni\n\n"
        f"KONTEKST DZISIAJ:\n{context}\n"
        + history_block +
        "\nRaport (format markdown z naglowkami sekcji **bold**):"
    )
    try:
        response = client.messages.create(
            model=LLM_MODEL,
            max_tokens=max_tokens,
            system=_LLM_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": prompt}],
        )
        text = response.content[0].text.strip()
        _save_commentary(
            section=section, chart_name="Final auction report",
            prompt=prompt, response=text,
            input_tokens=getattr(response.usage, "input_tokens", None),
            output_tokens=getattr(response.usage, "output_tokens", None),
        )
        return text
    except Exception as exc:
        return f"⚠️ _LLM error: {exc}_"


print(f"Connected. START_DATE = {START_DATE_STR}, TODAY = {TODAY_STR}")
print(f"LLM commentary: {'enabled (' + LLM_MODEL + ', history N=' + str(HISTORY_N) + ')' if _get_llm_client() else 'disabled (no API key)'}")


## 1. Skład długu BGK PLN — stacked area per program

In [ ]:
# Event-driven outstanding timeline (v_bgk_outstanding_timeline robi cumsum
# delty na poziomie SQL, per program × currency).
timeline = fetch_view(
    "v_bgk_outstanding_timeline",
    "?currency=eq.PLN&select=*&order=event_date.asc,program.asc",
)
timeline["event_date"] = pd.to_datetime(timeline["event_date"])
timeline["running_outstanding"] = pd.to_numeric(timeline["running_outstanding"], errors="coerce")

# Per program total events count + latest
diag = timeline[timeline["event_date"] <= TODAY]
src_stats = diag.groupby("program").agg(
    n_events=("event_date", "count"),
    first_date=("event_date", "min"),
    last_date=("event_date", "max"),
    last_outstanding=("running_outstanding", "last"),
).sort_values("last_outstanding", ascending=False)
print("Source events per program (with cumulative latest <= TODAY):")
print(src_stats)

# Pivot to wide (per-day index) - reindex+ffill per program
data_min = timeline["event_date"].min()
all_dates = pd.date_range(start=data_min, end=TODAY, freq="D")

wide = pd.DataFrame(index=all_dates)
for prog in sorted(timeline["program"].dropna().unique()):
    sub = timeline[timeline["program"] == prog].sort_values("event_date")
    s = sub.set_index("event_date")["running_outstanding"]
    s = s[~s.index.duplicated(keep="last")]
    s = s[s.index <= TODAY]
    wide[prog] = s.reindex(all_dates).ffill().fillna(0)

# Trim do START_DATE
wide = wide[wide.index >= START_DATE]

# Reduce do dni z faktycznymi zmianami
mask = (wide != wide.shift(1)).any(axis=1)
wide = wide[mask]

# Order programy od najwiekszego sredniego outstanding
order = wide.mean().sort_values(ascending=False).index.tolist()
wide = wide[order]
piv = wide

print(f"\nEvent dates: {len(piv)}, range: {piv.index.min().date()} → {piv.index.max().date()}")
print(f"Latest total: {piv.iloc[-1].sum() / 1000:.2f} bln PLN")
print(f"Latest per program: {dict((k, f'{v/1000:.1f}bln') for k, v in piv.iloc[-1].items())}")
print(f"Programs: {order}")
piv.tail()


In [ ]:
# Kolory per program - FPC dominantny niebieski, KFD pomaranczowy, etc.
PROGRAM_COLORS = {
    "FPC":    "#1f77b4",   # niebieski - dominantny
    "KFD":    "#ff7f0e",   # pomaranczowy
    "FP":     "#2ca02c",   # zielony - Fundusz Pracy
    "FWSZ":   "#d62728",   # czerwony - Sily Zbrojne
    "własne": "#9467bd",   # fioletowy - emisje wlasne
    "FWA":    "#8c564b",   # brazowy - FWA (historyczny)
}

fig = go.Figure()
for prog in order:
    color = PROGRAM_COLORS.get(prog, "grey")
    fig.add_trace(go.Scatter(
        x=piv.index, y=piv[prog] / 1000.0,
        name=prog, mode="lines", stackgroup="one",
        line=dict(color=color, width=0.5),
        hovertemplate="%{y:.2f} bln PLN<extra>" + prog + "</extra>",
    ))

totals_bln = piv.sum(axis=1) / 1000.0
fig.add_trace(go.Scatter(
    x=piv.index, y=totals_bln,
    name="Σ TOTAL", mode="lines",
    line=dict(color="rgba(0,0,0,0)", width=0),
    hovertemplate="<b>Σ TOTAL</b>: %{y:.2f} bln PLN<extra></extra>",
    showlegend=False,
    hoverlabel=dict(bgcolor="black", font=dict(color="white")),
))

fig.update_layout(
    title="BGK PLN — skład długu per program (event-driven)",
    xaxis_title="Data emisji / wykupu",
    yaxis_title="Outstanding (bln PLN)",
    template="plotly_white",
    hovermode="x unified",
    height=600,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

# LLM commentary
_latest = piv.iloc[-1]
_total = _latest.sum() / 1000
_top3 = _latest.sort_values(ascending=False).head(3)
_year_ago = piv[piv.index <= piv.index.max() - pd.Timedelta(days=365)]
_yoy = ((_latest.sum() / _year_ago.iloc[-1].sum()) - 1) * 100 if len(_year_ago) else 0
_summary = (
    f"Latest total ({_latest.name.date()}): {_total:.2f} bln PLN\n"
    f"Top programs: " + ", ".join(f"{p}={v/1000:.2f}bln" for p, v in _top3.items()) + "\n"
    f"YoY change in total: {_yoy:+.1f}%\n"
    f"Active programs: {(_latest > 0).sum()} / {len(_latest)}"
)
display(Markdown(llm_chart_commentary(
    "Chart 1 — Skład długu BGK PLN per program (stacked area)",
    "Stacked area cumulative outstanding per program BGK (FPC dominantny, KFD, FP, FWSZ, własne emisje) w PLN, event-driven. Pokazuje rampe FPC od marca 2020 i programowe diversification mix.",
    _summary,
)))


## 2. Skład długu BGK — udział procentowy per typ kuponu (stałe / zmienne)

In [ ]:
# Outstanding timeline per coupon_kind - musimy zbudowac w Pythonie bo SQL
# view agreguje per program × currency, nie per coupon_kind.
# Algorytm: events = (issue_date, +amount) + (maturity_date, -amount), per coupon_kind.

# Bierzemy wszystkie PLN emisje + ich coupon_kind
df_emit = fetch_view(
    "bgk_auctions",
    "?currency=eq.PLN&select=issue_date,maturity_date,coupon_kind,issue_amount&order=issue_date.asc",
)
df_emit["issue_date"] = pd.to_datetime(df_emit["issue_date"])
df_emit["maturity_date"] = pd.to_datetime(df_emit["maturity_date"])
df_emit["issue_amount"] = pd.to_numeric(df_emit["issue_amount"], errors="coerce")
df_emit["coupon_kind"] = df_emit["coupon_kind"].fillna("?")

# Build events
ev_iss = df_emit[["issue_date", "coupon_kind", "issue_amount"]].copy()
ev_iss.columns = ["event_date", "coupon_kind", "delta"]
ev_mat = df_emit[["maturity_date", "coupon_kind", "issue_amount"]].copy()
ev_mat["issue_amount"] = -ev_mat["issue_amount"]
ev_mat.columns = ["event_date", "coupon_kind", "delta"]
events = pd.concat([ev_iss, ev_mat], ignore_index=True)
events = events.sort_values("event_date")

# Cumsum per coupon_kind
piv_ck = events.pivot_table(
    index="event_date", columns="coupon_kind", values="delta", aggfunc="sum"
).fillna(0).cumsum().clip(lower=0)

# Daily index + ffill, trim to <=TODAY
all_dates = pd.date_range(start=piv_ck.index.min(), end=TODAY, freq="D")
piv_ck = piv_ck.reindex(all_dates).ffill().fillna(0)
piv_ck = piv_ck[piv_ck.index >= START_DATE]

# Reduce to change-only rows
mask = (piv_ck != piv_ck.shift(1)).any(axis=1)
piv_ck = piv_ck[mask]

# Procenty per row
row_sums = piv_ck.sum(axis=1)
piv_pct = piv_ck.div(row_sums.replace(0, pd.NA), axis=0) * 100
piv_pct = piv_pct.fillna(0)

# Order rysowania: stale na dole (zwykle wieksze)
KIND_ORDER = ["stałe", "zmienne"]
piv_pct = piv_pct[[c for c in KIND_ORDER if c in piv_pct.columns]]
piv_ck = piv_ck[piv_pct.columns]

print(f"Kinds: {list(piv_pct.columns)}")
print(f"Latest total: {row_sums.iloc[-1] / 1000:.2f} bln PLN")
print(f"Latest % per kind: {dict((k, f'{v:.1f}%') for k, v in piv_pct.iloc[-1].items())}")
print(f"Latest nominal per kind (bln): {dict((k, f'{v/1000:.2f}') for k, v in piv_ck.iloc[-1].items())}")
piv_pct.tail()


In [ ]:
KIND_COLORS = {
    "stałe":   "#1f77b4",   # niebieski - fixed
    "zmienne": "#ff7f0e",   # pomaranczowy - WIBOR floater
}

fig = go.Figure()
totals_bln = row_sums / 1000.0
totals_bln_change = totals_bln.loc[piv_pct.index]

for kind in piv_pct.columns:
    nominal_bln = piv_ck[kind] / 1000.0
    fig.add_trace(go.Scatter(
        x=piv_pct.index, y=piv_pct[kind],
        customdata=nominal_bln,
        name=kind, mode="lines", stackgroup="one",
        line=dict(width=0.5, color=KIND_COLORS.get(kind, "grey")),
        hovertemplate=("%{y:.1f}%  (%{customdata:.2f} bln PLN)"
                       "<extra>" + kind + "</extra>"),
    ))

fig.add_trace(go.Scatter(
    x=piv_pct.index, y=[100.0] * len(piv_pct),
    customdata=totals_bln_change,
    name="Σ TOTAL", mode="lines",
    line=dict(color="rgba(0,0,0,0)", width=0),
    hovertemplate="<b>Σ TOTAL</b>: %{customdata:.2f} bln PLN<extra></extra>",
    showlegend=False,
    hoverlabel=dict(bgcolor="black", font=dict(color="white")),
))

fig.update_layout(
    title="BGK PLN — udział % per typ kuponu (stałe vs zmienne)",
    xaxis_title="Data",
    yaxis_title="Udział (%)",
    yaxis=dict(range=[0, 100], ticksuffix="%"),
    template="plotly_white",
    hovermode="x unified",
    height=600,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

# LLM commentary
_latest_pct = piv_pct.iloc[-1]
_yr_ago = piv_pct[piv_pct.index <= piv_pct.index.max() - pd.Timedelta(days=365)]
_pct_lines = [f"  {k}: {_latest_pct[k]:.1f}%" for k in piv_pct.columns]
_shift_lines = []
if len(_yr_ago):
    _prev = _yr_ago.iloc[-1]
    for k in piv_pct.columns:
        delta = _latest_pct[k] - _prev.get(k, 0)
        _shift_lines.append(f"  {k}: {delta:+.1f}pp")
_summary = (
    "Latest mix:\n" + "\n".join(_pct_lines) +
    "\n\nYoY shift in mix:\n" + "\n".join(_shift_lines)
)
display(Markdown(llm_chart_commentary(
    "Chart 2 — Struktura % per coupon kind (stałe/zmienne)",
    "Stacked 100% area: udziały stałe vs zmienne w portfelu BGK PLN. Pokazuje strukturalna preferencja BGK na fixed/floater rate (różnie niż MF gdzie WZ/NZ to ~10%) - BGK FPC był heavy na zmienne podczas inflacji 2022-23.",
    _summary,
)))


## 3. Maturity ladder BGK — refinancing profile (Bloomberg DDIS style)

In [ ]:
# Wszystkie aktywne emisje PLN: issue_date<=TODAY, maturity_date>TODAY.
# Outstanding per ISIN = sum(issue_amount) - bo cala emisja zostanie wykupiona
# w maturity_date (BGK nie ma rolling buybackow w naszych danych).
active = df_emit[
    (df_emit["issue_date"] <= TODAY)
    & (df_emit["maturity_date"] > TODAY)
].copy()
active["maturity_year"] = active["maturity_date"].dt.year

# Per (year, coupon_kind)
agg = active.groupby(["maturity_year", "coupon_kind"], as_index=False)["issue_amount"].sum()
pivot_mat = agg.pivot(index="maturity_year", columns="coupon_kind", values="issue_amount").fillna(0)
KIND_ORDER_LADDER = ["stałe", "zmienne"]
pivot_mat = pivot_mat[[k for k in KIND_ORDER_LADDER if k in pivot_mat.columns]].sort_index()

# Stats
yearly_totals = pivot_mat.sum(axis=1)
total_outstanding = yearly_totals.sum()
peak_year = yearly_totals.idxmax()
peak_val = yearly_totals.max()

# WAM (years)
active["yrs_to_mat"] = (active["maturity_date"] - TODAY).dt.days / 365.25
wam_yrs = (active["yrs_to_mat"] * active["issue_amount"]).sum() / active["issue_amount"].sum()

# Next 12M maturities
mat_12m = active[active["maturity_date"] <= TODAY + pd.Timedelta(days=365)]["issue_amount"].sum()

# Matured YTD (issue_amount summed for maturities in current year before today)
all_emit_full = df_emit.copy()
year_start = pd.Timestamp(year=TODAY.year, month=1, day=1)
mat_ytd_rows = all_emit_full[
    (all_emit_full["maturity_date"] >= year_start)
    & (all_emit_full["maturity_date"] <= TODAY)
]
matured_by_kind = mat_ytd_rows.groupby("coupon_kind")["issue_amount"].sum()
matured_ytd_total = matured_by_kind.sum() if len(matured_by_kind) else 0.0

print(f"Total outstanding ({TODAY.date()}): {total_outstanding/1000:.2f} bln PLN")
print(f"WAM: {wam_yrs:.2f} lat")
print(f"Peak year: {peak_year} ({peak_val/1000:.2f} bln PLN)")
print(f"Next 12M maturities: {mat_12m/1000:.2f} bln PLN ({100*mat_12m/total_outstanding:.1f}% portfela)")
print(f"\nMatured YTD {TODAY.year}: {matured_ytd_total/1000:.2f} bln PLN")
if len(matured_by_kind):
    for k, v in matured_by_kind.items():
        print(f"   {k}: {v/1000:.2f} bln")
print(f"\nKind totals (REMAINING - bln PLN):")
for k in pivot_mat.columns:
    print(f"  {k}: {pivot_mat[k].sum()/1000:.2f}")


In [ ]:
fig = go.Figure()

current_year = TODAY.year
# Matured YTD jako hatched bar
for kind in KIND_ORDER_LADDER:
    if kind in matured_by_kind.index and matured_by_kind[kind] > 0:
        color = KIND_COLORS.get(kind, "grey")
        fig.add_trace(go.Bar(
            x=[current_year], y=[matured_by_kind[kind]],
            name=f"{kind} matured YTD",
            marker=dict(color=color, pattern=dict(shape="/", fgcolor="white", size=4)),
            opacity=0.55, showlegend=True,
            legendgroup=f"matured_{kind}",
            hovertemplate=(f"<b>{kind} MATURED YTD {current_year}</b><br>"
                           "%{y:,.0f} mln PLN<extra></extra>"),
        ))

# Remaining bars per (year, kind)
for kind in pivot_mat.columns:
    fig.add_trace(go.Bar(
        x=pivot_mat.index, y=pivot_mat[kind],
        name=kind,
        marker_color=KIND_COLORS.get(kind, "grey"),
        hovertemplate=(f"<b>{kind}</b><br>%{{x}}: %{{y:,.0f}} mln PLN<extra></extra>"),
    ))

# Total annotations
for year in sorted(set(pivot_mat.index).union({current_year} if matured_ytd_total > 0 else set())):
    remaining = yearly_totals.get(year, 0)
    if year == current_year and matured_ytd_total > 0:
        total_for_year = remaining + matured_ytd_total
        label = f"<b>{total_for_year/1000:.2f}</b><br><span style='color:#666;font-size:9px'>(rem {remaining/1000:.2f} + mat {matured_ytd_total/1000:.2f})</span>"
    else:
        total_for_year = remaining
        label = f"<b>{total_for_year/1000:.2f}</b>"
    if total_for_year <= 0:
        continue
    fig.add_annotation(
        x=year, y=total_for_year, text=label, showarrow=False, yshift=14,
        font=dict(size=10, color="black"),
    )

fig.add_annotation(
    x=peak_year, y=0, text=f"PEAK<br>{peak_year}",
    showarrow=False, yshift=-20,
    font=dict(size=10, color="red", family="Arial Black"),
)

callout = (
    f"<b>Total outstanding: {total_outstanding/1000:.2f} bln PLN</b><br>"
    f"WAM: {wam_yrs:.2f} lat<br>"
    f"Next 12M: {mat_12m/1000:.2f} bln "
    f"({100*mat_12m/total_outstanding:.1f}% portfela)<br>"
    f"Matured YTD {current_year}: {matured_ytd_total/1000:.2f} bln"
)
fig.add_annotation(
    xref="paper", yref="paper", x=0.02, y=0.98, xanchor="left", yanchor="top",
    text=callout, showarrow=False, align="left",
    bgcolor="white", bordercolor="black", borderwidth=1, font=dict(size=11),
)

fig.update_layout(
    title=(f"Maturity ladder BGK PLN (snapshot {TODAY.date()}) — "
           "solid = remaining, hatched = matured YTD"),
    xaxis_title="Rok wykupu", yaxis_title="Outstanding (mln PLN)",
    barmode="stack", template="plotly_white", height=600,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
    xaxis=dict(dtick=1),
)
fig.show()

# LLM commentary
_top3_yrs = yearly_totals.sort_values(ascending=False).head(3)
_summary = (
    f"Snapshot {TODAY.date()}:\n"
    f"  Total outstanding: {total_outstanding/1000:.2f} bln PLN\n"
    f"  WAM: {wam_yrs:.2f} lat\n"
    f"  Peak refinancing year: {peak_year} ({peak_val/1000:.2f} bln)\n"
    f"  Next 12M: {mat_12m/1000:.2f} bln ({100*mat_12m/total_outstanding:.1f}% portfela)\n"
    f"  Matured YTD {current_year}: {matured_ytd_total/1000:.2f} bln\n\n"
    f"Top 3 peak years: " +
    ", ".join(f"{y}={v/1000:.2f}bln" for y, v in _top3_yrs.items())
)
display(Markdown(llm_chart_commentary(
    "Chart 3 — Maturity ladder BGK (DDIS style)",
    "Stacked bar per rok wykupu, kolor per coupon kind. Pokazuje 'wall of refinancing' BGK - gdzie sa skoncentrowane wykupy. WAM (weighted avg maturity) + Next-12M w callout. Hatched portion na current year = co juz zostalo zrolowane YTD.",
    _summary,
)))


## 4. Aukcje BGK FPC — statystyki per dzień (B/C + spread vs POLGB)

In [ ]:
# v_bgk_auction_metrics ma per-series B/C + NK shares. v_bgk_auction_spread
# ma per-series spread vs POLGB. Joinujemy + agregujemy per auction_date.

df_metrics = fetch_view(
    "v_bgk_auction_metrics",
    f"?auction_date=gte.{START_DATE_STR}&select=*&order=auction_date.asc",
)
df_metrics["auction_date"] = pd.to_datetime(df_metrics["auction_date"])
for c in ["demand_total_mln", "demand_nc_mln", "sold_total_mln", "sold_nc_mln",
          "bid_to_cover", "allocation_rate", "nc_share_demand", "nc_share_sold",
          "stop_price", "yield_pct", "total_taken_mln"]:
    if c in df_metrics.columns:
        df_metrics[c] = pd.to_numeric(df_metrics[c], errors="coerce")

df_spread = fetch_view(
    "v_bgk_auction_spread",
    f"?auction_date=gte.{START_DATE_STR}&select=auction_date,series,spread_bp"
    "&order=auction_date.asc",
)
df_spread["auction_date"] = pd.to_datetime(df_spread["auction_date"])
df_spread["spread_bp"] = pd.to_numeric(df_spread["spread_bp"], errors="coerce")

# Per-day aggregat
agg = df_metrics.groupby("auction_date", as_index=False).agg(
    total_sold_mln=("sold_total_mln", "sum"),
    total_demand_mln=("demand_total_mln", "sum"),
    total_nk_mln=("demand_nc_mln", "sum"),
    total_sold_nk_mln=("sold_nc_mln", "sum"),
)
agg["bid_to_cover"] = agg["total_demand_mln"] / agg["total_sold_mln"]
agg["nc_share_demand"] = agg["total_nk_mln"] / agg["total_demand_mln"]

# Spread aggregated per auction_date (mean across series tego dnia, weighted
# byloby lepiej ale brakuje wagi - approximate)
spread_per_day = df_spread.groupby("auction_date", as_index=False)["spread_bp"].mean()
agg = agg.merge(spread_per_day, on="auction_date", how="left")

MA_WINDOW = 12
agg = agg.sort_values("auction_date").reset_index(drop=True)
agg["bid_to_cover_ma"] = agg["bid_to_cover"].rolling(window=MA_WINDOW, min_periods=4).mean()

print(f"Auction days: {len(agg)},  range: {agg.auction_date.min().date()} → {agg.auction_date.max().date()}")
print(f"Z spreadem: {agg['spread_bp'].notna().sum()}/{len(agg)} aukcji")


In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=("Bid-to-cover + total sold",
                                    "Spread vs POLGB (bp)"),
                    specs=[[{"secondary_y": True}], [{}]])

fig.add_trace(go.Bar(x=agg["auction_date"], y=agg["total_sold_mln"],
                     name="Total sold (mln PLN)", marker_color="lightgrey",
                     opacity=0.6), row=1, col=1, secondary_y=True)
fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["bid_to_cover"],
                         name="Bid-to-cover", mode="lines+markers",
                         line=dict(color="navy", width=1.5),
                         marker=dict(size=4),
                         opacity=0.55),
              row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["bid_to_cover_ma"],
                         name=f"B/C MA({MA_WINDOW} aukcji)",
                         mode="lines",
                         line=dict(color="darkred", width=2.5)),
              row=1, col=1, secondary_y=False)

fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["spread_bp"],
                         name="Spread BGK − POLGB (bp)", mode="lines+markers",
                         line=dict(color="seagreen", width=1.5)), row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="grey", row=2, col=1)

fig.update_yaxes(title_text="B/C", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="mln PLN", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="bp", row=2, col=1)

fig.update_layout(
    title="Statystyki aukcji BGK FPC (PDF) — B/C + spread vs POLGB",
    template="plotly_white", height=700,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.10),
)
fig.show()

# LLM commentary
_latest = agg.iloc[-1]
_recent = agg.tail(26)
_summary = (
    f"Latest auction ({_latest.auction_date.date()}):\n"
    f"  total sold {_latest.total_sold_mln:,.0f} mln, "
    f"B/C {_latest.bid_to_cover:.2f}, "
    f"spread {_latest.spread_bp:+.1f}bp\n"
    f"MA({MA_WINDOW}) B/C: {_latest.bid_to_cover_ma:.2f}\n\n"
    f"Last 6m (n={len(_recent)}): "
    f"B/C mean {_recent.bid_to_cover.mean():.2f} ±{_recent.bid_to_cover.std():.2f}, "
    f"spread mean {_recent.spread_bp.mean():+.1f}bp"
)
display(Markdown(llm_chart_commentary(
    "Chart 4 — B/C + spread vs POLGB w czasie",
    "Time series aukcji BGK FPC (PDF komunikaty): bid-to-cover z MA12 jako structural demand trend + spread BGK−POLGB w bp. Pokazuje ogólną sile/slabosc popytu na agencyjny dług FPC i premie placona vs POLGB jako proxy market access cost.",
    _summary,
)))


## 5. B/C i spread per typ kuponu (stałe / zmienne)

In [ ]:
# v_bgk_auction_spread MV ma bgk_coupon_kind. Joinujemy z df_metrics zeby
# dostac B/C per series, potem aggregujemy per (date, coupon_kind).

df_spread_full = fetch_view(
    "v_bgk_auction_spread",
    f"?auction_date=gte.{START_DATE_STR}&select=auction_date,series,isin,"
    "bgk_coupon_kind,spread_bp&order=auction_date.asc",
)
df_spread_full["auction_date"] = pd.to_datetime(df_spread_full["auction_date"])
df_spread_full["spread_bp"] = pd.to_numeric(df_spread_full["spread_bp"], errors="coerce")

# Join with df_metrics per (auction_date, series) to bring B/C
df_per_series = df_spread_full.merge(
    df_metrics[["auction_date", "series", "bid_to_cover", "sold_total_mln",
                "demand_total_mln", "nc_share_demand"]],
    on=["auction_date", "series"], how="left",
)

# Aggregate per (auction_date, coupon_kind) - weighted by sold_total_mln
def _wavg(df, col, wcol="sold_total_mln"):
    w = df[wcol].fillna(0)
    v = df[col].astype(float)
    if w.sum() == 0:
        return v.mean()
    return (v * w).sum() / w.sum()

sub5 = df_per_series.dropna(subset=["bgk_coupon_kind"]).copy()
agg5 = sub5.groupby(["auction_date", "bgk_coupon_kind"]).apply(
    lambda g: pd.Series({
        "bid_to_cover": _wavg(g, "bid_to_cover"),
        "spread_bp":    _wavg(g, "spread_bp"),
        "total_sold_mln": g["sold_total_mln"].sum(),
        "n_series":     len(g),
    }),
    include_groups=False,
).reset_index().rename(columns={"bgk_coupon_kind": "coupon_kind"})

print(f"Rows: {len(agg5)}, kinds: {sorted(agg5.coupon_kind.unique())}")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=("Bid-to-cover", "Spread vs POLGB (bp)"))

for kind in sorted(agg5.coupon_kind.unique()):
    sub = agg5[agg5.coupon_kind == kind].sort_values("auction_date")
    color = KIND_COLORS.get(kind, "grey")
    fig.add_trace(go.Scatter(x=sub["auction_date"], y=sub["bid_to_cover"],
                             name=kind, legendgroup=kind, mode="lines+markers",
                             line=dict(color=color, width=1.5), marker=dict(size=4)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=sub["auction_date"], y=sub["spread_bp"],
                             name=kind, legendgroup=kind, showlegend=False,
                             mode="lines+markers",
                             line=dict(color=color, width=1.5), marker=dict(size=4)),
                  row=2, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="grey", row=2, col=1)
fig.update_layout(
    title="Metryki aukcyjne per typ kuponu (BGK FPC)",
    template="plotly_white", height=700,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.10),
)
fig.update_yaxes(title_text="B/C", row=1, col=1)
fig.update_yaxes(title_text="bp", row=2, col=1)
fig.show()

# LLM commentary
_5_latest_date = agg5["auction_date"].max()
_5_latest = agg5[agg5.auction_date == _5_latest_date]
_5_lines = []
for _, r in _5_latest.iterrows():
    _5_lines.append(
        f"  {r.coupon_kind}: B/C {r.bid_to_cover:.2f}, "
        f"spread {r.spread_bp:+.1f}bp, sold {r.total_sold_mln:,.0f} mln, "
        f"series n={int(r.n_series)}"
    )
_summary = (
    f"Latest auction date {_5_latest_date.date()}:\n" +
    "\n".join(_5_lines)
)
display(Markdown(llm_chart_commentary(
    "Chart 5 — B/C + spread per coupon kind (stałe/zmienne)",
    "Time series B/C i spread BGK−POLGB rozbite per coupon kind (stałe vs zmienne). Pozwala porównać popyt na fixed vs floater BGK. Floatery (zmienne) korzystają z WIBOR floor podczas wysokich rates - sprawdza czy spread DM jest stabilny vs spread yield-space.",
    _summary,
)))


## 6. Aukcje — popyt, podaż, wykonanie (per series, FPC PDF)

In [ ]:
# Stacked demand (competitive + NK) + sold per auction date.
# Brak offer_min/max w PDF (BGK nie publikuje pre-auction announced range),
# wiec ograniczamy do "popyt vs sprzedaz" (allocation).

day_da = agg[["auction_date", "total_demand_mln", "total_nk_mln",
              "total_sold_mln", "total_sold_nk_mln",
              "bid_to_cover", "nc_share_demand"]].copy().sort_values("auction_date")
day_da["comp_demand"] = day_da["total_demand_mln"] - day_da["total_nk_mln"]
day_da["comp_sold"] = day_da["total_sold_mln"] - day_da["total_sold_nk_mln"]
day_da["allocation_rate"] = day_da["total_sold_mln"] / day_da["total_demand_mln"]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    row_heights=[0.65, 0.35],
                    subplot_titles=("Popyt (stacked: competitive + NK) vs sold (overlay)",
                                    "Allocation rate (sold / demand)"))

fig.add_trace(go.Bar(x=day_da["auction_date"], y=day_da["comp_demand"],
                     name="Competitive demand", marker_color="lightsteelblue",
                     hovertemplate="%{x|%Y-%m-%d}<br>comp: %{y:,.0f} mln<extra></extra>"),
              row=1, col=1)
fig.add_trace(go.Bar(x=day_da["auction_date"], y=day_da["total_nk_mln"],
                     name="NK demand", marker_color="darkorange",
                     hovertemplate="%{x|%Y-%m-%d}<br>NK: %{y:,.0f} mln<extra></extra>"),
              row=1, col=1)

fig.add_trace(go.Scatter(x=day_da["auction_date"], y=day_da["total_sold_mln"],
                         name="Total sold", mode="lines+markers",
                         line=dict(color="darkred", width=2),
                         marker=dict(size=5),
                         hovertemplate="%{x|%Y-%m-%d}<br>sold: %{y:,.0f} mln<extra></extra>"),
              row=1, col=1)

# Allocation rate as bars
colors_alloc = ["#2ca02c" if v < 0.5 else "#ff7f0e" if v < 1 else "#d62728"
                for v in day_da["allocation_rate"].fillna(0)]
fig.add_trace(go.Bar(x=day_da["auction_date"], y=day_da["allocation_rate"] * 100,
                     marker_color=colors_alloc, name="Allocation %",
                     showlegend=False,
                     hovertemplate="%{x|%Y-%m-%d}<br>%{y:.1f}%<extra></extra>"),
              row=2, col=1)

fig.update_yaxes(title_text="mln PLN", row=1, col=1)
fig.update_yaxes(title_text="%", row=2, col=1, range=[0, 110])
fig.update_layout(
    title="BGK FPC — popyt vs sprzedaż per aukcja",
    barmode="stack", template="plotly_white", height=750,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.10),
)
fig.show()

# LLM commentary
_6_latest = day_da.iloc[-1]
_summary = (
    f"Latest auction ({_6_latest.auction_date.date()}):\n"
    f"  Total demand: {_6_latest.total_demand_mln:,.0f} mln "
    f"(comp: {_6_latest.comp_demand:,.0f}, NK: {_6_latest.total_nk_mln:,.0f})\n"
    f"  Total sold: {_6_latest.total_sold_mln:,.0f} mln "
    f"(allocation rate {_6_latest.allocation_rate*100:.1f}%)\n"
    f"  B/C: {_6_latest.bid_to_cover:.2f}, NK share: {_6_latest.nc_share_demand*100:.1f}%"
)
display(Markdown(llm_chart_commentary(
    "Chart 6 — Popyt vs sprzedaż BGK FPC",
    "Stacked bars per aukcja: competitive + NK demand, overlay = total sold. Drugi panel: allocation rate. <100% = rationing (oversubscribed = strong demand); =100% = full take.",
    _summary,
)))


## 7. Historyczna dystrybucja per seria — box ploty

In [ ]:
# Ostatnia aukcja: serie sprzedane. Box plots dla kazdej z tych serii
# pelnej ich historii B/C i spread.
latest_auction_date = df_per_series["auction_date"].max()
df_last = df_per_series[df_per_series["auction_date"] == latest_auction_date].copy()
today_series_list = sorted(df_last["series"].dropna().unique().tolist())

df_hist = df_per_series[
    df_per_series["series"].isin(today_series_list)
    & (df_per_series["auction_date"] < latest_auction_date)
].copy()

print(f"Today's series ({latest_auction_date.date()}): {today_series_list}")
print(f"Total historical auctions of these series: {len(df_hist)}")
for s in today_series_list:
    n = len(df_hist[df_hist["series"] == s])
    print(f"  {s}: {n} prior auctions")


In [ ]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
    subplot_titles=(f"Historyczne Bid-to-cover per seria (od {START_DATE.year})",
                    f"Historyczne Spread bp per seria (od {START_DATE.year})"),
)

today_marker = dict(
    symbol="diamond", size=14, color="red",
    line=dict(color="black", width=1),
)

for i, s in enumerate(today_series_list):
    hist = df_hist[df_hist.series == s]
    today_row = df_last[df_last.series == s].iloc[0]
    today_bc = pd.to_numeric(today_row.get("bid_to_cover"), errors="coerce")
    today_spread = pd.to_numeric(today_row.get("spread_bp"), errors="coerce")
    n_hist = len(hist)

    fig.add_trace(go.Box(
        y=hist["bid_to_cover"], name=s,
        marker_color="lightblue", boxmean=True,
        showlegend=False,
        hovertemplate=(
            f"<b>{s}</b> (n={n_hist})<br>"
            "%{y:.2f}<extra></extra>"
        ),
    ), row=1, col=1)
    if pd.notna(today_bc):
        fig.add_trace(go.Scatter(
            x=[s], y=[today_bc],
            mode="markers", marker=today_marker,
            name="dzisiaj", legendgroup="today",
            showlegend=(i == 0),
            hovertemplate=f"<b>{s}</b><br>dzisiaj B/C: %{{y:.2f}}<extra></extra>",
        ), row=1, col=1)

    fig.add_trace(go.Box(
        y=hist["spread_bp"], name=s,
        marker_color="lightgreen", boxmean=True,
        showlegend=False,
        hovertemplate=(
            f"<b>{s}</b> (n={n_hist})<br>"
            "%{y:+.1f} bp<extra></extra>"
        ),
    ), row=2, col=1)
    if pd.notna(today_spread):
        fig.add_trace(go.Scatter(
            x=[s], y=[today_spread],
            mode="markers", marker=today_marker,
            name="dzisiaj", legendgroup="today",
            showlegend=False,
            hovertemplate=f"<b>{s}</b><br>dzisiaj spread: %{{y:+.1f}} bp<extra></extra>",
        ), row=2, col=1)

fig.add_hline(y=1, line_dash="dot", line_color="grey", row=1, col=1)
fig.add_hline(y=0, line_dash="dot", line_color="grey", row=2, col=1)

fig.update_layout(
    title=f"Aukcja {latest_auction_date.date()}: dzisiaj vs pełna historia per seria",
    template="plotly_white",
    height=750,
    showlegend=True,
    legend=dict(orientation="h", y=-0.10),
)
fig.update_yaxes(title_text="B/C", row=1, col=1)
fig.update_yaxes(title_text="bp", row=2, col=1)
fig.update_xaxes(title_text="Seria", row=2, col=1)
fig.show()

# LLM commentary
_7_lines = []
for s in today_series_list:
    hist = df_hist[df_hist.series == s]
    today_row = df_last[df_last.series == s].iloc[0]
    bc_today = pd.to_numeric(today_row.get("bid_to_cover"), errors="coerce")
    spread_today = pd.to_numeric(today_row.get("spread_bp"), errors="coerce")
    bc_med = hist["bid_to_cover"].median() if len(hist) else float("nan")
    spread_med = hist["spread_bp"].median() if len(hist) else float("nan")
    _7_lines.append(
        f"  {s} (n={len(hist)}): "
        f"B/C today {bc_today:.2f} vs median {bc_med:.2f}, "
        f"spread today {spread_today:+.1f}bp vs median {spread_med:+.1f}bp"
    )
_summary = (
    f"Aukcja {latest_auction_date.date()} — serie vs pelna historia:\n" +
    "\n".join(_7_lines)
)
display(Markdown(llm_chart_commentary(
    "Chart 7 — Box plots per seria z dzisiejszą aukcją",
    "Per każda seria dzisiaj sprzedana: box plot wszystkich poprzednich aukcji tej serii (B/C i spread bp) + czerwony diament dzisiaj. Pokazuje czy print był typowy, czy outlier w kontekście historycznym konkretnej serii BGK.",
    _summary,
)))


## 8. Tail metric — spread per aukcja z kolorowaniem (BGK FPC)

In [ ]:
# Per-aukcja spread BGK vs POLGB. Konwencja jak concession CETO:
# spread > 0 = BGK plat wiecej niz POLGB = tail (czerwony)
# spread < 0 = BGK plat mniej niz POLGB = through / strong demand (zielony)
df8 = agg[["auction_date", "spread_bp", "total_sold_mln"]].dropna(
    subset=["spread_bp"]
).copy().sort_values("auction_date").reset_index(drop=True)

WIN_8 = 6
df8["roll_mean"] = df8["spread_bp"].rolling(WIN_8, min_periods=3).mean()
df8["roll_std"] = df8["spread_bp"].rolling(WIN_8, min_periods=3).std()

print(f"Auctions ze spreadem: {len(df8)}, "
      f"range {df8.auction_date.min().date()} → {df8.auction_date.max().date()}")
print(f"Latest spread: {df8.iloc[-1]['spread_bp']:+.1f} bp "
      f"(roll mean {df8.iloc[-1]['roll_mean']:+.1f} ±{df8.iloc[-1]['roll_std']:.1f})")

fig = go.Figure()

# ±1 SD band
fig.add_trace(go.Scatter(
    x=df8["auction_date"], y=df8["roll_mean"] + df8["roll_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"),
    showlegend=False, hoverinfo="skip",
))
fig.add_trace(go.Scatter(
    x=df8["auction_date"], y=df8["roll_mean"] - df8["roll_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"),
    fill="tonexty", fillcolor="rgba(120,120,120,0.18)",
    name=f"±1 SD ({WIN_8}-auction)",
    hoverinfo="skip",
))

# Bary kolorowane: < 0 = through (zielone), >= 0 = tail (czerwone)
colors_8 = ["#2ca02c" if v < 0 else "#d62728" for v in df8["spread_bp"]]
fig.add_trace(go.Bar(
    x=df8["auction_date"], y=df8["spread_bp"],
    marker_color=colors_8,
    name="Spread per aukcja",
    hovertemplate=(
        "%{x|%Y-%m-%d}<br>"
        "spread: %{y:+.1f} bp<br>"
        "sold: %{customdata:,.0f} mln<extra></extra>"
    ),
    customdata=df8["total_sold_mln"],
))

# Rolling mean line
fig.add_trace(go.Scatter(
    x=df8["auction_date"], y=df8["roll_mean"],
    mode="lines", line=dict(color="black", width=2),
    name=f"Średnia {WIN_8}-aukcji",
))

fig.add_hline(y=0, line_dash="dash", line_color="grey")

fig.update_layout(
    title="Tail / spread per aukcja — zielony = through (BGK<POLGB), czerwony = tail (BGK>POLGB)",
    xaxis_title="Data aukcji",
    yaxis_title="Spread BGK − POLGB (bp)",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

# LLM commentary
_8_latest = df8.iloc[-1]
_8_recent = df8.tail(WIN_8 * 2)
_through_n = (_8_recent["spread_bp"] < 0).sum()
_tail_n = (_8_recent["spread_bp"] > 0).sum()
_z = ((_8_latest.spread_bp - _8_latest.roll_mean) / _8_latest.roll_std
      if _8_latest.roll_std and _8_latest.roll_std > 0 else 0)
_summary = (
    f"Latest auction ({_8_latest.auction_date.date()}): "
    f"spread {_8_latest.spread_bp:+.1f} bp; "
    f"MA({WIN_8}) mean {_8_latest.roll_mean:+.1f} ±{_8_latest.roll_std:.1f}\n"
    f"Z-score dzisiejszego print: {_z:+.2f}σ\n\n"
    f"Last {len(_8_recent)} aukcji: {_through_n} through (zielony), {_tail_n} tail (czerwony)"
)
display(Markdown(llm_chart_commentary(
    "Chart 8 — Tail / spread per aukcja BGK",
    "Bar chart per aukcja: zielone = stops through (BGK<POLGB, silny popyt), czerwone = tail (BGK>POLGB, koncesja). Linia MA(6) + szare pasmo ±1SD pokazuje typowy zakres - outliery sygnalizują exceptional prints.",
    _summary,
)))


## 9. Auction scorecard — dzisiejsza aukcja vs rolling 6m / 12m

In [ ]:
# Per-bucket (coupon_kind) scorecard: B/C, spread_bp, NK share
# vs rolling 6m i 12m mean/std + z-score.

sc_per_series = df_per_series.copy()
sc_per_series["bucket"] = sc_per_series["bgk_coupon_kind"].fillna("?")

# Per (date, bucket) - weighted by sold
def _wavg(df, col, wcol="sold_total_mln"):
    w = df[wcol].fillna(0)
    v = df[col].astype(float)
    if w.sum() == 0:
        return v.mean()
    return (v * w).sum() / w.sum()

sc_agg = sc_per_series.groupby(["auction_date", "bucket"]).apply(
    lambda g: pd.Series({
        "bid_to_cover":  _wavg(g, "bid_to_cover"),
        "spread_bp":     _wavg(g, "spread_bp"),
        "nk_share_pct":  _wavg(g, "nc_share_demand") * 100,
        "sold":          g["sold_total_mln"].sum(),
    }),
    include_groups=False,
).reset_index()

if sc_agg.empty or sc_agg["bucket"].dropna().empty:
    print("! sc_agg empty - brak danych do scorecard")
else:
    latest_date_sc = sc_agg["auction_date"].max()
    WIN_6M, WIN_12M = 26, 52
    # (column, label, sign +1=higher better, -1=lower better, hint)
    metric_specs = [
        ("bid_to_cover",  "B/C",            +1, "↑ better"),
        ("spread_bp",     "Spread bp",      -1, "↓ better"),
        ("nk_share_pct",  "NK share %",     +1, "↑ better"),
    ]
    metric_sign = {label: sign for _, label, sign, _ in metric_specs}
    metric_hint = {label: hint for _, label, _, hint in metric_specs}

    records = []
    for bucket in sorted(sc_agg["bucket"].dropna().unique()):
        sub = sc_agg[sc_agg.bucket == bucket].sort_values("auction_date")
        today_rows = sub[sub.auction_date == latest_date_sc]
        if today_rows.empty:
            continue
        today_row = today_rows.iloc[0]
        hist_all = sub[sub.auction_date < latest_date_sc]
        for col, label, _, _ in metric_specs:
            today_val = today_row[col]
            if pd.isna(today_val):
                continue
            for win_label, win in [("6m", WIN_6M), ("12m", WIN_12M)]:
                hist = hist_all.tail(win)[col].dropna()
                if len(hist) < 3:
                    continue
                mean, std = hist.mean(), hist.std()
                z = (today_val - mean) / std if std > 0 else float("nan")
                records.append({
                    "bucket": bucket,
                    "metric": label,
                    "today": today_val,
                    "window": win_label,
                    "mean": mean,
                    "std": std,
                    "z_score": z,
                    "n_hist": len(hist),
                })

    scorecard = pd.DataFrame(records)
    print(f"Scorecard dla aukcji {latest_date_sc.date()}: {len(scorecard)} records, "
          f"{scorecard['bucket'].nunique() if not scorecard.empty else 0} buckets")

    if scorecard.empty:
        print("! Brak danych - dla zadnego bucketu nie mamy ≥3 historycznych aukcji.")
    else:
        pivoted = scorecard.pivot_table(
            index=["bucket", "metric"],
            columns="window",
            values=["mean", "z_score"],
            aggfunc="first",
        )
        today_vals = scorecard.groupby(["bucket", "metric"])["today"].first()
        out = pd.DataFrame(index=today_vals.index)
        out["dzisiaj"] = today_vals
        if ("mean", "6m") in pivoted.columns:
            out["śr 6m"] = pivoted[("mean", "6m")]
            out["z (6m)"] = pivoted[("z_score", "6m")]
        if ("mean", "12m") in pivoted.columns:
            out["śr 12m"] = pivoted[("mean", "12m")]
            out["z (12m)"] = pivoted[("z_score", "12m")]
        out = out.reset_index()
        out["kierunek"] = out["metric"].map(metric_hint)
        col_order = ["bucket", "metric", "kierunek", "dzisiaj"]
        for c in ["śr 6m", "z (6m)", "śr 12m", "z (12m)"]:
            if c in out.columns:
                col_order.append(c)
        out = out[col_order]

        def _color_z(row):
            metric = row["metric"]
            sign = metric_sign.get(metric, +1)
            styles = [""] * len(row)
            for idx, col_name in enumerate(row.index):
                if col_name not in ("z (6m)", "z (12m)"):
                    continue
                z = row[col_name]
                if pd.isna(z):
                    continue
                favorable = z * sign
                a = abs(favorable)
                if a < 1:
                    continue
                if favorable >= 2:
                    styles[idx] = "background-color:#198754;color:white;font-weight:bold"
                elif favorable > 0:
                    styles[idx] = "background-color:#d1e7dd"
                elif favorable > -2:
                    styles[idx] = "background-color:#f8d7da"
                else:
                    styles[idx] = "background-color:#dc3545;color:white;font-weight:bold"
            return styles

        styled = out.style.format({
            "dzisiaj":  "{:+.2f}",
            "śr 6m":    "{:+.2f}",
            "z (6m)":   "{:+.2f}",
            "śr 12m":   "{:+.2f}",
            "z (12m)":  "{:+.2f}",
        }, na_rep="—").hide(axis="index").apply(_color_z, axis=1)
        display(styled)

        # LLM commentary
        _9_lines = []
        for _, r in out.iterrows():
            z6 = r.get("z (6m)")
            z6_str = f"z6m={z6:+.2f}" if pd.notna(z6) else "z6m=n/a"
            _9_lines.append(
                f"  {r.bucket} / {r.metric}: today {r.dzisiaj:+.2f}, "
                f"6m mean {r.get('śr 6m', float('nan')):+.2f}, {z6_str}"
            )
        _summary = (
            f"Scorecard dla aukcji {latest_date_sc.date()} (z-score vs rolling history):\n"
            + "\n".join(_9_lines) +
            "\n\nKierunek 'better': B/C up, Spread down (mniej premii), NK share up."
        )
        display(Markdown(llm_chart_commentary(
            "Chart 9 — Auction scorecard z z-scores",
            "Per coupon kind (stałe/zmienne), porównanie 3 metryk (B/C, Spread bp, NK share) dzisiaj vs rolling 6m/12m mean+std. Z-score >=1 lub <=-1 sygnalizuje odbieganie od typowego, >=2 lub <=-2 = ekstremum.",
            _summary,
        )))


## 10. NK (non-competitive) bidding — trend, udział, fill rate

In [ ]:
import numpy as np

day_nk = day_da[["auction_date", "total_demand_mln", "total_nk_mln",
                 "total_sold_nk_mln", "comp_demand"]].copy().sort_values("auction_date").reset_index(drop=True)
day_nk["nk_share_pct"] = day_nk["total_nk_mln"] / day_nk["total_demand_mln"].replace(0, np.nan) * 100
day_nk["nk_fill_pct"] = (
    day_nk["total_sold_nk_mln"] / day_nk["total_nk_mln"].replace(0, np.nan) * 100
).clip(upper=100).astype(float)

WIN_10 = 12
day_nk["roll_share"] = day_nk["nk_share_pct"].rolling(WIN_10, min_periods=4).mean()
day_nk["roll_share_std"] = day_nk["nk_share_pct"].rolling(WIN_10, min_periods=4).std()
day_nk["roll_fill"] = day_nk["nk_fill_pct"].rolling(WIN_10, min_periods=4).mean()
day_nk["roll_fill_std"] = day_nk["nk_fill_pct"].rolling(WIN_10, min_periods=4).std()

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    row_heights=[0.40, 0.30, 0.30],
    subplot_titles=(
        "Demand stacked: competitive (niebieski) + NK (pomarańczowy)",
        f"NK share % popytu — kropki + MA({WIN_10}) ±1 SD",
        f"NK fill rate % = sold_NK / demand_NK — <100% = rationowane (oversubscribed)",
    ),
)

fig.add_trace(go.Bar(
    x=day_nk["auction_date"], y=day_nk["comp_demand"],
    name="Competitive demand", marker_color="lightsteelblue",
    hovertemplate="%{x|%Y-%m-%d}<br>comp: %{y:,.0f} mln<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Bar(
    x=day_nk["auction_date"], y=day_nk["total_nk_mln"],
    name="NK demand", marker_color="darkorange",
    hovertemplate="%{x|%Y-%m-%d}<br>NK: %{y:,.0f} mln<extra></extra>",
), row=1, col=1)

# Row 2: NK share
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_share"] + day_nk["roll_share_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"), showlegend=False, hoverinfo="skip",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_share"] - day_nk["roll_share_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"),
    fill="tonexty", fillcolor="rgba(255,165,0,0.18)",
    name="±1 SD (share)", hoverinfo="skip",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["nk_share_pct"],
    mode="markers", marker=dict(color="darkorange", size=4, opacity=0.55),
    name="NK share %",
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:.1f}%<extra></extra>",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_share"],
    mode="lines", line=dict(color="darkorange", width=2.5),
    name=f"MA({WIN_10}) share",
), row=2, col=1)

# Row 3: NK fill rate
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_fill"] + day_nk["roll_fill_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"), showlegend=False, hoverinfo="skip",
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_fill"] - day_nk["roll_fill_std"],
    mode="lines", line=dict(color="rgba(0,0,0,0)"),
    fill="tonexty", fillcolor="rgba(150,80,200,0.18)",
    name="±1 SD (fill)", hoverinfo="skip",
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["nk_fill_pct"],
    mode="markers", marker=dict(color="#8B4FBA", size=4, opacity=0.55),
    name="NK fill %",
    hovertemplate="%{x|%Y-%m-%d}<br>fill: %{y:.1f}%<extra></extra>",
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=day_nk["auction_date"], y=day_nk["roll_fill"],
    mode="lines", line=dict(color="#8B4FBA", width=2.5),
    name=f"MA({WIN_10}) fill",
), row=3, col=1)
fig.add_hline(y=100, line_dash="dot", line_color="grey", row=3, col=1,
              annotation_text="100% = brak rationowania", annotation_position="top right")

fig.update_yaxes(title_text="mln PLN", row=1, col=1)
fig.update_yaxes(title_text="NK share (%)", row=2, col=1)
fig.update_yaxes(title_text="NK fill rate (%)", row=3, col=1, range=[0, 105])
fig.update_layout(
    title="BGK FPC — NK (non-competitive) bidding",
    template="plotly_white", barmode="stack",
    height=850, hovermode="x unified",
    legend=dict(orientation="h", y=-0.08),
)
fig.show()

if not day_nk.empty:
    latest = day_nk.iloc[-1]
    print(f"\\nLatest auction {latest['auction_date'].date()}:")
    print(f"  Total demand: {latest['total_demand_mln']:,.0f} mln "
          f"(comp: {latest['comp_demand']:,.0f}, NK: {latest['total_nk_mln']:,.0f})")
    print(f"  NK share:    {latest['nk_share_pct']:.1f}% "
          f"(MA{WIN_10}: {latest['roll_share']:.1f}% ±{latest['roll_share_std']:.1f})")
    if pd.notna(latest["nk_fill_pct"]):
        print(f"  NK fill rate: {latest['nk_fill_pct']:.1f}% "
              f"— MA{WIN_10}: {latest['roll_fill']:.1f}%")
    else:
        print(f"  NK fill rate: brak (demand_nk = 0)")

# LLM commentary
_10_latest = day_nk.iloc[-1] if len(day_nk) else None
_summary = "no data"
if _10_latest is not None:
    _summary = (
        f"Latest auction ({_10_latest.auction_date.date()}):\n"
        f"  Total demand: {_10_latest.total_demand_mln:,.0f} mln "
        f"(comp {_10_latest.comp_demand:,.0f} + NK {_10_latest.total_nk_mln:,.0f})\n"
        f"  NK share: {_10_latest.nk_share_pct:.1f}% (MA{WIN_10}: {_10_latest.roll_share:.1f}%)\n"
        f"  NK fill rate: {_10_latest.nk_fill_pct:.1f}% (MA{WIN_10}: {_10_latest.roll_fill:.1f}%)\n"
        f"  Fill <100% = NK pool rationowany (oversubscribed, end-user demand >cap)"
    )
display(Markdown(llm_chart_commentary(
    "Chart 10 — NK (non-competitive) bidding BGK",
    "3 panele: stacked competitive+NK demand, NK share % popytu z MA12, NK fill rate (sold/demand). Fill <100% = NK pool oversubscribed (silny real-money demand, cap binding); 100% = caly NK popyt zaspokojony. BGK NK to glównie banki krajowe i fundusze emerytalne.",
    _summary,
)))


---
## 11. Raport finalny — analiza dzisiejszej aukcji

In [ ]:
# Final report - aggregate context, push to LLM via llm_final_report().

# Per-series line dla ostatniej aukcji (bez tenoru - df_last nie ma maturity_date).
_per_series_lines = []
for _, r in df_last.iterrows():
    _per_series_lines.append(
        f"  {r.series} ({r.bgk_coupon_kind}): "
        f"sold {r.sold_total_mln:,.0f} mln, "
        f"B/C {r.bid_to_cover:.2f}, "
        f"spread {r.spread_bp:+.1f}bp, "
        f"NK {(r.nc_share_demand or 0)*100:.1f}%"
    )

_final_ctx_parts = [
    f"DZISIEJSZA DATA: {TODAY.date()}",
    "",
    "=== SKLAD DLUGU BGK PLN (chart 1) ===",
    f"Total outstanding: {piv.iloc[-1].sum()/1000:.2f} bln PLN",
] + [f"  {p}: {piv.iloc[-1][p]/1000:.2f} bln"
     for p in piv.columns if piv.iloc[-1][p] > 0] + [
    "",
    "=== STRUKTURA KUPONOWA (chart 2) ===",
] + [f"  {k}: {piv_pct.iloc[-1][k]:.1f}% ({piv_ck.iloc[-1][k]/1000:.2f} bln)"
     for k in piv_pct.columns] + [
    "",
    "=== MATURITY LADDER (chart 3) ===",
    f"Total outstanding: {total_outstanding/1000:.2f} bln, WAM {wam_yrs:.2f}Y",
    f"Peak year: {peak_year} ({peak_val/1000:.2f} bln)",
    f"Next 12M maturities: {mat_12m/1000:.2f} bln "
    f"({100*mat_12m/total_outstanding:.1f}% portfela)",
    f"Matured YTD {TODAY.year}: {matured_ytd_total/1000:.2f} bln",
    "",
    "=== OSTATNIA AUKCJA FPC (chart 4-7) ===",
    f"Data: {latest_auction_date.date()}",
    f"Total sold: {_latest.total_sold_mln:,.0f} mln, "
    f"total demand: {_latest.total_demand_mln:,.0f} mln, "
    f"B/C overall: {_latest.bid_to_cover:.2f}",
    f"Serie ({len(df_last)}): " + ", ".join(df_last["series"].tolist()),
    "",
    "Per seria detail:",
] + _per_series_lines + [
    "",
    "=== TAIL / SPREAD CONTEXT (chart 8) ===",
    f"Latest spread: {df8.iloc[-1].spread_bp:+.1f} bp "
    f"(MA{WIN_8} {df8.iloc[-1].roll_mean:+.1f} +/-{df8.iloc[-1].roll_std:.1f})",
    f"Z-score: {((df8.iloc[-1].spread_bp - df8.iloc[-1].roll_mean) / df8.iloc[-1].roll_std if df8.iloc[-1].roll_std > 0 else 0):+.2f}sigma",
    "",
    "=== NK BIDDING (chart 10) ===",
    f"Latest NK share: {day_nk.iloc[-1].nk_share_pct:.1f}% (MA{WIN_10} {day_nk.iloc[-1].roll_share:.1f}%)",
    f"Latest NK fill rate: {day_nk.iloc[-1].nk_fill_pct:.1f}% "
    f"(MA{WIN_10} {day_nk.iloc[-1].roll_fill:.1f}%) - <100% = oversubscribed",
]

_final_context = "\n".join(p for p in _final_ctx_parts if p is not None)

display(Markdown(llm_final_report(_final_context)))
print(f"\n_Raport wygenerowany przez {LLM_MODEL} o {pd.Timestamp.now(tz='UTC').strftime('%Y-%m-%d %H:%M UTC')}_")
